# Perturbation selection stability and functional similarity

Paper: Fig. 5g and Supplementary Fig. S8b. Inputs: original full-cohort feature table, final trained PeriClock artifact and external pathway libraries. Environment: Python/Scanpy, NumPy, pandas, SciPy, scikit-learn, joblib, statsmodels, Matplotlib and GSEApy. Outputs: per-bootstrap selections, success logs, selection frequencies and Jaccard tables, functional permutation CSV and plots under `OUTPUT_DIR/bootstrap_stability`. All intermediate files are generated earlier in this notebook; no previous tutorial memory is needed. No bootstrap optimization or permutation analysis was executed during preparation.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Shared full-cohort feature matrix, final PeriClock model, GO BP 2023/Reactome 2022 libraries and licensed Arial fonts; [training matrices](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices) and [other references](../RESOURCE_DOWNLOAD.md#8-other-reference-databases).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
FEATURES_1000 = Path(os.environ.get("PERICLOCK_FEATURES_1000", str(RESOURCE_DIR / "datasets/PeriClock-1000s_PeriClock_features_matrix.pkl"))).expanduser().resolve()
MODEL_PATH = Path(os.environ.get("PERICLOCK_PERTURBATION_MODEL", str(MODEL_DIR / "PeriClock.pkl"))).expanduser().resolve()
GMT_DIR = Path(os.environ.get("PERICLOCK_GMT_DIR", str(RESOURCE_DIR / "external/enrichr"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/07_perturbation_stability"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [FEATURES_1000, MODEL_PATH]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Original feature preparation and bootstrap comparison

Source: cells 0, 2–3, 5, 7–11. Cell 5 provides the two original solvers; cell 7 replaces the bootstrap wrapper before its first invocation, preserving the research execution order. Each age group has 50 bootstraps; frequencies are normalized by successful runs. Inspect `bootstrap_run_summary.csv` and failure messages before interpreting stability. The original cell 2 refits variance/feature selection and scaling on the full feature table; this behavior is retained, rather than silently substituted with the saved transforms used in Tutorials 05–06. Its numerical equivalence to the saved transforms has not been rechecked. This is expensive and was not executed.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
import scanpy as sc
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

In [ ]:
df_all_features = joblib.load(str(FEATURES_1000))

pipeline = joblib.load(str(MODEL_PATH))
vt = pipeline['vt']
skb = pipeline['skb']
scaler_prop = pipeline['scaler_prop']
scaler_expr = pipeline['scaler_expr']
en_model = pipeline['model']
prop_cols = pipeline['prop_cols']
expr_cols = pipeline['expr_cols']
selected_features = pipeline['selected_features']

df = df_all_features[selected_features + ['Age']]

# 目标变量 y 和特征矩阵 X
y = df_all_features['Age'].values
X = df_all_features.drop(columns='Age')

# 区分细胞比例特征（假设以 "_prop" 结尾）和表达特征（其余）
prop_cols = [col for col in X.columns if col.endswith('_ratio')]
expr_cols = [col for col in X.columns if col not in prop_cols and col != 'Individual']
expr_cols

X_prop = X[prop_cols].copy()
X_expr = X[expr_cols].copy()

# 去除低方差表达特征
vt = VarianceThreshold(threshold=1e-5)
X_expr_vt = vt.fit_transform(X_expr)

# 选择最相关的前 k 个特征
skb = SelectKBest(score_func=f_regression, k=8000)   # k=8000
X_expr_selected = skb.fit_transform(X_expr_vt, y)

# 获取保留下来的列名
selected_expr_cols = X_expr.columns[vt.get_support()][skb.get_support()]

# 合并，标准化
scaler_prop = StandardScaler()
scaler_expr = StandardScaler()

X_prop_scaled = scaler_prop.fit_transform(X_prop)
X_expr_scaled = scaler_expr.fit_transform(X_expr_selected)

# 合并，带列名
all_cols = list(X_prop.columns) + list(selected_expr_cols)
X_final = np.hstack([X_prop_scaled, X_expr_scaled])
X_final_df = pd.DataFrame(X_final, columns=all_cols, index=df_all_features.index)
X = X_final_df[selected_features]


In [ ]:
X = X.astype(np.float64)

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize
from itertools import combinations

# ============================================================
# 1. Cell-type-specific：标准化空间下界
# ============================================================
def build_scaled_lower_bounds(feature_names, scaler_expr, selected_expr_cols, scaler_prop=None, prop_cols=None, min_expr=0.0, min_prop=0.0):
    lower = pd.Series(index=feature_names, dtype=float)
    expr_mean, expr_scale = pd.Series(scaler_expr.mean_, index=selected_expr_cols), pd.Series(scaler_expr.scale_, index=selected_expr_cols)
    expr_lower = (min_expr - expr_mean) / expr_scale
    for f in feature_names:
        if f in expr_lower.index: lower[f] = expr_lower[f]
    if scaler_prop is not None and prop_cols is not None:
        prop_mean, prop_scale = pd.Series(scaler_prop.mean_, index=prop_cols), pd.Series(scaler_prop.scale_, index=prop_cols)
        prop_lower = (min_prop - prop_mean) / prop_scale
        for f in feature_names:
            if f in prop_lower.index: lower[f] = prop_lower[f]
    return lower

# ============================================================
# 2. Cell-type-specific LP
# ============================================================
def solve_shared_rejuvenation_design(X_target, en_model, scaler_expr, selected_expr_cols, scaler_prop=None, prop_cols=None, reduce_years=10, candidate_features=None, max_change=1.0, min_expr=0.0, min_prop=0.0, weight_by_sd=True, perturbation_tol=1e-6, age_effect_tol=0.01):
    if candidate_features is None: candidate_features = list(X_target.columns)
    candidate_features = [f for f in candidate_features if f in X_target.columns]

    beta_all = pd.Series(en_model.coef_, index=X_target.columns, dtype=float)
    beta = beta_all.loc[candidate_features]
    keep = beta.abs() > 1e-12
    candidate_features, beta = beta.index[keep].tolist(), beta.loc[keep]
    if not candidate_features: raise ValueError("candidate_features中没有非零ElasticNet系数特征。")

    X0 = X_target[candidate_features].copy()
    age_before = np.asarray(en_model.predict(X_target), dtype=float)
    lower_scaled = build_scaled_lower_bounds(candidate_features, scaler_expr, selected_expr_cols, scaler_prop, prop_cols, min_expr, min_prop)

    if weight_by_sd:
        sd = X0.std().replace(0, np.nan)
        weights = (1 / sd).replace([np.inf, -np.inf], np.nan)
        weights = pd.Series(1.0, index=candidate_features) if weights.notna().sum() == 0 else weights.fillna(weights.median()).clip(lower=weights.quantile(0.05), upper=weights.quantile(0.95))
    else:
        weights = pd.Series(1.0, index=candidate_features)

    n = len(candidate_features)
    c = np.concatenate([weights.values, weights.values])
    A_ub = np.concatenate([beta.values, -beta.values]).reshape(1, -1)
    b_ub = np.array([-reduce_years])

    bounds = [(0, max_change)] * n
    for f in candidate_features:
        max_down = X0[f].min() if pd.isna(lower_scaled[f]) else (X0[f] - lower_scaled[f]).min()
        bounds.append((0, max(0, min(max_change, max_down))))

    res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    if not res.success: return None

    d = res.x[:n] - res.x[n:]
    X_full_new = X_target.copy()
    X_full_new.loc[:, candidate_features] = X0.values + d
    age_after_full = np.asarray(en_model.predict(X_full_new), dtype=float)

    full_df = pd.DataFrame({
        "feature": candidate_features, "beta": beta.values, "perturbation_scaled": d,
        "abs_perturbation_scaled": np.abs(d), "age_effect": beta.values * d,
        "mean_x_scaled_original": X0.mean().values, "mean_x_scaled_new": X_full_new[candidate_features].mean().values,
        "lower_bound_scaled": lower_scaled.values, "weight": weights.values
    })
    full_df["direction"] = np.where(full_df["perturbation_scaled"] > 0, "up", "down")

    result_df = full_df[(full_df["abs_perturbation_scaled"] > perturbation_tol) & (full_df["age_effect"].abs() > age_effect_tol)].copy()

    X_new = X_target.copy()
    if len(result_df):
        core = result_df["feature"].tolist()
        d_core = pd.Series(d, index=candidate_features).loc[core]
        X_new.loc[:, core] = X_target.loc[:, core] + d_core.values

    age_after_core = np.asarray(en_model.predict(X_new), dtype=float)

    summary = {
        "success": True, "n_samples": len(X_target), "n_features_core": len(result_df),
        "mean_age_reduction_full": float(np.mean(age_before - age_after_full)),
        "mean_age_reduction_core": float(np.mean(age_before - age_after_core)),
        "weighted_total_perturbation": float((result_df["abs_perturbation_scaled"] * result_df["weight"]).sum())
    }
    return result_df, summary, full_df

# ============================================================
# 3. feature -> gene映射
# ============================================================
def build_feature_gene_map(selected_features, selected_expr_cols):
    expr_set = set(map(str, selected_expr_cols))
    features = [str(f) for f in selected_features if str(f) in expr_set]
    return pd.Series({f: f.rsplit("_", 1)[1].upper() for f in features}, name="Gene")

# ============================================================
# 4. Gene-level统一FC扰动
# ============================================================
def solve_uniform_fold_change_rejuvenation(X_target, en_model, scaler_expr, selected_expr_cols, feature_gene_map, reduce_years=10, candidate_genes=None, min_fold_change=0.2, max_fold_change=2.0, constraint_mode="mean", perturbation_tol=0.01, age_effect_tol=0.01, gene_weights=None, maxiter=500, ftol=1e-6):
    if constraint_mode not in ["mean", "all"]: raise ValueError("constraint_mode必须是'mean'或'all'。")

    feature_gene_map = pd.Series(feature_gene_map).copy()
    expression_features = [f for f in selected_expr_cols if f in X_target.columns and f in feature_gene_map.index]
    if not expression_features: raise ValueError("没有可用于统一比例扰动的表达特征。")

    feature_gene_map = feature_gene_map.loc[expression_features].astype(str)
    beta_all = pd.Series(en_model.coef_, index=X_target.columns, dtype=float)
    beta_expr = beta_all.loc[expression_features]
    keep = beta_expr.abs() > 1e-12
    expression_features, beta_expr = beta_expr.index[keep].tolist(), beta_expr.loc[keep]
    feature_gene_map = feature_gene_map.loc[expression_features]
    if not expression_features: raise ValueError("表达特征ElasticNet系数全部为0。")

    available_genes = sorted(feature_gene_map.unique())
    genes = available_genes if candidate_genes is None else [g for g in available_genes if g in set(map(str, candidate_genes))]
    if not genes: raise ValueError("没有可用于优化的候选基因。")

    expression_features = feature_gene_map.index[feature_gene_map.isin(genes)].tolist()
    feature_gene_map, beta_expr = feature_gene_map.loc[expression_features], beta_expr.loc[expression_features]

    expr_mean = pd.Series(scaler_expr.mean_, index=selected_expr_cols, dtype=float).loc[expression_features]
    expr_scale = pd.Series(scaler_expr.scale_, index=selected_expr_cols, dtype=float).loc[expression_features]
    if (expr_scale <= 0).any(): raise ValueError("scaler_expr.scale_中存在非正值。")

    X_log1p = (X_target[expression_features] * expr_scale + expr_mean).clip(lower=0)
    X_linear = np.expm1(X_log1p).replace([np.inf, -np.inf], np.nan)
    if X_linear.isna().any().any(): raise ValueError("表达值反变换后出现NaN或无穷值。")

    age_before = np.asarray(en_model.predict(X_target), dtype=float)
    n_genes = len(genes)
    gene_to_idx = {g: i for i, g in enumerate(genes)}
    feature_gene_idx = np.array([gene_to_idx[feature_gene_map.loc[f]] for f in expression_features])

    if gene_weights is None: weight = np.ones(n_genes)
    else: weight = pd.Series(gene_weights, dtype=float).reindex(genes).fillna(1.0).values

    max_up, max_down = np.log(max_fold_change), abs(np.log(min_fold_change))
    bounds = [(0, max_up)] * n_genes + [(0, max_down)] * n_genes

    def apply_fc(z):
        log_fc = z[:n_genes] - z[n_genes:]
        fc = np.exp(log_fc)
        feature_fc = fc[feature_gene_idx]
        new_linear = X_linear.values * feature_fc[np.newaxis, :]
        new_log1p = np.log1p(new_linear)
        new_scaled = (new_log1p - expr_mean.values[np.newaxis, :]) / expr_scale.values[np.newaxis, :]
        X_new = X_target.copy()
        X_new.loc[:, expression_features] = new_scaled
        return X_new, fc, log_fc

    def objective(z): return np.sum(weight * (z[:n_genes] + z[n_genes:]))

    def constraint(z):
        X_new, _, _ = apply_fc(z)
        reduction = age_before - np.asarray(en_model.predict(X_new), dtype=float)
        return reduction.mean() - reduce_years if constraint_mode == "mean" else reduction - reduce_years

    res = minimize(objective, np.zeros(2 * n_genes), method="SLSQP", bounds=bounds, constraints=[{"type": "ineq", "fun": constraint}], options={"maxiter": maxiter, "ftol": ftol, "disp": False})
    if not res.success: return {"success": False, "optimization_result": res}

    X_full, fc_full, log_fc_full = apply_fc(res.x)
    age_after_full = np.asarray(en_model.predict(X_full), dtype=float)
    fc_s, log_fc_s = pd.Series(fc_full, index=genes), pd.Series(log_fc_full, index=genes)

    gene_effect = {}
    for gene in genes:
        fc = fc_s.loc[gene]
        if abs(fc - 1) <= 1e-12:
            gene_effect[gene] = 0.0
            continue
        features = feature_gene_map[feature_gene_map == gene].index.tolist()
        old_log = X_log1p[features]
        new_log = np.log1p(np.expm1(old_log) * fc)
        X_single = X_target.copy()
        X_single.loc[:, features] = (new_log - expr_mean.loc[features]) / expr_scale.loc[features]
        gene_effect[gene] = float(np.mean(age_before - en_model.predict(X_single)))

    gene_rows = []
    for gene in genes:
        fc, log_fc = fc_s.loc[gene], log_fc_s.loc[gene]
        gene_rows.append({
            "gene": gene, "fold_change": fc, "log_fold_change": log_fc, "percent_change": (fc - 1) * 100,
            "abs_percent_change": abs((fc - 1) * 100), "direction": "up" if fc > 1 else ("down" if fc < 1 else "none"),
            "mean_age_reduction_contribution": gene_effect[gene],
            "n_celltype_features": int((feature_gene_map == gene).sum()),
            "weight": weight[gene_to_idx[gene]], "weighted_abs_log_fold_change": weight[gene_to_idx[gene]] * abs(log_fc)
        })

    full_gene_df = pd.DataFrame(gene_rows).sort_values("mean_age_reduction_contribution", ascending=False)
    gene_df = full_gene_df[(full_gene_df["fold_change"].sub(1).abs() > perturbation_tol) & (full_gene_df["mean_age_reduction_contribution"].abs() > age_effect_tol)].copy()

    return {
        "success": True, "gene_perturbation_df": gene_df, "full_gene_result_df": full_gene_df,
        "summary": {"n_samples": len(X_target), "n_genes_core": len(gene_df), "mean_age_reduction_full": float(np.mean(age_before - age_after_full)), "optimization_objective": float(res.fun)},
        "optimization_result": res
    }

# ============================================================
# 5. Bootstrap稳定性
# ============================================================
def run_bootstrap_stability(X, en_model, scaler_expr, selected_expr_cols, scaler_prop, prop_cols, selected_features, age_range=(50, 60), n_boot=50, reduce_years=10, seed=2026, output_dir=None):
    if output_dir is None: output_dir = f"bootstrap_stability_{age_range[0]}-{age_range[1]}"
    os.makedirs(output_dir, exist_ok=True)

    pred = pd.Series(en_model.predict(X), index=X.index)
    X_target = X.loc[(pred >= age_range[0]) & (pred < age_range[1])].copy()
    if X_target.empty: raise ValueError(f"{age_range}没有目标样本。")

    gene_map = build_feature_gene_map(selected_features, selected_expr_cols)
    model_expr_features = set(gene_map.index)
    rng = np.random.default_rng(seed)

    cell_rows, gene_rows, run_rows = [], [], []

    print(f"Target age: {age_range[0]}-{age_range[1]}, N={len(X_target)}, bootstrap={n_boot}")

    for b in range(n_boot):
        idx = rng.integers(0, len(X_target), len(X_target))
        X_boot = X_target.iloc[idx].copy().astype(np.float64)
        cell_success = gene_success = False
        n_cf = n_cg = n_gg = 0

        try:
            out = solve_shared_rejuvenation_design(X_boot, en_model, scaler_expr, selected_expr_cols, scaler_prop, prop_cols, reduce_years=reduce_years, candidate_features=None, max_change=1.0, min_expr=0.0, min_prop=0.0, weight_by_sd=True, perturbation_tol=1e-6, age_effect_tol=0.01)
            if out is not None:
                cdf = out[0].copy()
                cdf["feature"] = cdf["feature"].astype(str)
                expr_df = cdf[cdf["feature"].isin(model_expr_features)].copy()
                expr_df["gene"] = expr_df["feature"].map(gene_map).astype(str).str.upper()

                for f in expr_df["feature"].unique(): cell_rows.append({"bootstrap": b, "level": "feature", "target": f})
                for g in expr_df["gene"].unique(): cell_rows.append({"bootstrap": b, "level": "gene", "target": g})

                cell_success, n_cf, n_cg = True, expr_df["feature"].nunique(), expr_df["gene"].nunique()
        except Exception as e:
            print(f"\nCell-specific bootstrap {b+1} failed: {e}")

        try:
            gout = solve_uniform_fold_change_rejuvenation(X_boot, en_model, scaler_expr, selected_expr_cols, gene_map, reduce_years=reduce_years, candidate_genes=None, min_fold_change=0.2, max_fold_change=2.0, constraint_mode="mean", perturbation_tol=0.01, age_effect_tol=0.01, maxiter=500, ftol=1e-6)
            if gout["success"]:
                gdf = gout["gene_perturbation_df"].copy()
                gdf["gene"] = gdf["gene"].astype(str).str.upper()
                for g in gdf["gene"].unique(): gene_rows.append({"bootstrap": b, "target": g})
                gene_success, n_gg = True, gdf["gene"].nunique()
        except Exception as e:
            print(f"\nGene-level bootstrap {b+1} failed: {e}")

        run_rows.append({"bootstrap": b, "cell_success": cell_success, "gene_success": gene_success, "n_cell_features": n_cf, "n_cell_genes": n_cg, "n_gene_level_genes": n_gg})
        print(f"\rBootstrap {b+1}/{n_boot}", end="")

    print("\nFinished.")

    cell_records, gene_records, run_df = pd.DataFrame(cell_rows), pd.DataFrame(gene_rows), pd.DataFrame(run_rows)
    n_cell_success, n_gene_success = int(run_df["cell_success"].sum()), int(run_df["gene_success"].sum())

    def selection_frequency(df, denom, level=None):
        if level is not None: df = df[df["level"] == level]
        if df.empty or denom == 0: return pd.DataFrame(columns=["target", "selection_frequency"])
        return df.groupby("target")["bootstrap"].nunique().div(denom).sort_values(ascending=False).rename("selection_frequency").reset_index()

    cell_feature_freq = selection_frequency(cell_records, n_cell_success, "feature").rename(columns={"target": "feature"})
    cell_gene_freq = selection_frequency(cell_records, n_cell_success, "gene").rename(columns={"target": "gene"})
    gene_level_freq = selection_frequency(gene_records, n_gene_success).rename(columns={"target": "gene"})

    between_rows = []
    for b in range(n_boot):
        rr = run_df.loc[run_df["bootstrap"] == b].iloc[0]
        if not (rr["cell_success"] and rr["gene_success"]): continue
        cs = set(cell_records[(cell_records["bootstrap"] == b) & (cell_records["level"] == "gene")]["target"])
        gs = set(gene_records[gene_records["bootstrap"] == b]["target"])
        u = cs | gs
        between_rows.append({"bootstrap": b, "n_cell_genes": len(cs), "n_gene_genes": len(gs), "n_shared": len(cs & gs), "jaccard": len(cs & gs) / len(u) if u else np.nan})

    between_df = pd.DataFrame(between_rows)

    def internal_jaccard(records, successful_boots, level=None):
        if level is not None: records = records[records["level"] == level]
        sets = {b: set(records.loc[records["bootstrap"] == b, "target"]) for b in successful_boots}
        rows = []
        for b1, b2 in combinations(successful_boots, 2):
            a, c = sets[b1], sets[b2]
            u = a | c
            rows.append({"bootstrap_1": b1, "bootstrap_2": b2, "jaccard": len(a & c) / len(u) if u else np.nan})
        return pd.DataFrame(rows)

    cell_boots = run_df.loc[run_df["cell_success"], "bootstrap"].tolist()
    gene_boots = run_df.loc[run_df["gene_success"], "bootstrap"].tolist()
    cell_internal = internal_jaccard(cell_records, cell_boots, "gene")
    gene_internal = internal_jaccard(gene_records, gene_boots)

    run_df.to_csv(os.path.join(output_dir, "bootstrap_run_summary.csv"), index=False)
    cell_feature_freq.to_csv(os.path.join(output_dir, "cell_specific_feature_selection_frequency.csv"), index=False)
    cell_gene_freq.to_csv(os.path.join(output_dir, "cell_specific_gene_selection_frequency.csv"), index=False)
    gene_level_freq.to_csv(os.path.join(output_dir, "gene_level_gene_selection_frequency.csv"), index=False)
    between_df.to_csv(os.path.join(output_dir, "between_strategy_bootstrap_jaccard.csv"), index=False)
    cell_internal.to_csv(os.path.join(output_dir, "cell_specific_internal_jaccard.csv"), index=False)
    gene_internal.to_csv(os.path.join(output_dir, "gene_level_internal_jaccard.csv"), index=False)

    print("\n===== Success =====")
    print(f"Cell-specific: {n_cell_success}/{n_boot}")
    print(f"Gene-level:    {n_gene_success}/{n_boot}")

    print("\n===== Mean Jaccard =====")
    print(f"Cell-specific internal: {cell_internal['jaccard'].mean():.3f}")
    print(f"Gene-level internal:    {gene_internal['jaccard'].mean():.3f}")
    print(f"Between strategies:     {between_df['jaccard'].mean():.3f}")

    print("\n===== Cell-specific top genes =====")
    print(cell_gene_freq.head(20).to_string(index=False))

    print("\n===== Gene-level top genes =====")
    print(gene_level_freq.head(20).to_string(index=False))

    return {
        "run_summary": run_df, "cell_feature_frequency": cell_feature_freq, "cell_gene_frequency": cell_gene_freq,
        "gene_level_frequency": gene_level_freq, "between_jaccard": between_df,
        "cell_internal_jaccard": cell_internal, "gene_internal_jaccard": gene_internal
    }

In [ ]:
# ============================================================
# Bootstrap stability：同时保存feature-level和gene-level结果
# ============================================================
def run_bootstrap_stability(X, en_model, scaler_expr, selected_expr_cols, scaler_prop, prop_cols, selected_features, age_range=(50, 60), n_boot=50, reduce_years=10, seed=2026, output_dir=None):
    if output_dir is None: output_dir = f"bootstrap_stability_{age_range[0]}-{age_range[1]}"
    os.makedirs(output_dir, exist_ok=True)

    pred = pd.Series(en_model.predict(X), index=X.index)
    X_target = X.loc[(pred >= age_range[0]) & (pred < age_range[1])].copy().astype(float)
    if X_target.empty: raise ValueError(f"{age_range}没有目标样本。")

    gene_map = build_feature_gene_map(selected_features, selected_expr_cols)
    model_expr_features = set(gene_map.index)
    rng = np.random.default_rng(seed)

    cell_rows, gene_rows, run_rows = [], [], []
    print(f"Target age: {age_range[0]}-{age_range[1]}, N={len(X_target)}, bootstrap={n_boot}")

    for b in range(n_boot):
        idx = rng.integers(0, len(X_target), len(X_target))
        X_boot = X_target.iloc[idx].copy().astype(float)
        cell_success = gene_success = False
        n_cf = n_cg = n_gg = 0

        try:
            out = solve_shared_rejuvenation_design(X_boot, en_model, scaler_expr, selected_expr_cols, scaler_prop, prop_cols, reduce_years=reduce_years, candidate_features=None, max_change=1.0, min_expr=0.0, min_prop=0.0, weight_by_sd=True, perturbation_tol=1e-6, age_effect_tol=0.01)
            if out is not None:
                cdf = out[0].copy()
                cdf["feature"] = cdf["feature"].astype(str)
                cdf = cdf[cdf["feature"].isin(model_expr_features)].copy()
                cdf["gene"] = cdf["feature"].map(gene_map).astype(str).str.upper()

                for f in cdf["feature"].unique(): cell_rows.append({"bootstrap": b, "level": "feature", "target": f})
                for g in cdf["gene"].unique(): cell_rows.append({"bootstrap": b, "level": "gene", "target": g})

                cell_success, n_cf, n_cg = True, cdf["feature"].nunique(), cdf["gene"].nunique()
        except Exception as e:
            print(f"\nCell-specific bootstrap {b+1} failed: {e}")

        try:
            gout = solve_uniform_fold_change_rejuvenation(X_boot, en_model, scaler_expr, selected_expr_cols, gene_map, reduce_years=reduce_years, candidate_genes=None, min_fold_change=0.2, max_fold_change=2.0, constraint_mode="mean", perturbation_tol=0.01, age_effect_tol=0.01, maxiter=500, ftol=1e-6)
            if gout["success"]:
                gdf = gout["gene_perturbation_df"].copy()
                gdf["gene"] = gdf["gene"].astype(str).str.upper()
                for g in gdf["gene"].unique(): gene_rows.append({"bootstrap": b, "target": g})
                gene_success, n_gg = True, gdf["gene"].nunique()
        except Exception as e:
            print(f"\nGene-level bootstrap {b+1} failed: {e}")

        run_rows.append({"bootstrap": b, "cell_success": cell_success, "gene_success": gene_success, "n_cell_features": n_cf, "n_cell_genes": n_cg, "n_gene_level_genes": n_gg})
        print(f"\rBootstrap {b+1}/{n_boot}", end="")

    print("\nFinished.")

    cell_records = pd.DataFrame(cell_rows, columns=["bootstrap", "level", "target"])
    gene_records = pd.DataFrame(gene_rows, columns=["bootstrap", "target"])
    run_df = pd.DataFrame(run_rows)

    n_cell_success = int(run_df["cell_success"].sum())
    n_gene_success = int(run_df["gene_success"].sum())
    cell_boots = run_df.loc[run_df["cell_success"], "bootstrap"].tolist()
    gene_boots = run_df.loc[run_df["gene_success"], "bootstrap"].tolist()

    def selection_frequency(df, denom, level=None):
        if level is not None: df = df[df["level"] == level]
        if df.empty or denom == 0: return pd.DataFrame(columns=["target", "selection_frequency"])
        return df.groupby("target")["bootstrap"].nunique().div(denom).sort_values(ascending=False).rename("selection_frequency").reset_index()

    def internal_jaccard(records, boots, level=None):
        if level is not None: records = records[records["level"] == level]
        sets = {b: set(records.loc[records["bootstrap"] == b, "target"]) for b in boots}
        rows = []
        for b1, b2 in combinations(boots, 2):
            a, c = sets[b1], sets[b2]
            u = a | c
            rows.append({"bootstrap_1": b1, "bootstrap_2": b2, "jaccard": len(a & c) / len(u) if u else np.nan})
        return pd.DataFrame(rows)

    cell_feature_freq = selection_frequency(cell_records, n_cell_success, "feature").rename(columns={"target": "feature"})
    cell_gene_freq = selection_frequency(cell_records, n_cell_success, "gene").rename(columns={"target": "gene"})
    gene_level_freq = selection_frequency(gene_records, n_gene_success).rename(columns={"target": "gene"})

    cell_feature_internal = internal_jaccard(cell_records, cell_boots, "feature")
    cell_gene_internal = internal_jaccard(cell_records, cell_boots, "gene")
    gene_level_internal = internal_jaccard(gene_records, gene_boots)

    between_rows = []
    for b in sorted(set(cell_boots) & set(gene_boots)):
        cs = set(cell_records[(cell_records["bootstrap"] == b) & (cell_records["level"] == "gene")]["target"])
        gs = set(gene_records[gene_records["bootstrap"] == b]["target"])
        u = cs | gs
        between_rows.append({"bootstrap": b, "n_cell_genes": len(cs), "n_gene_genes": len(gs), "n_shared": len(cs & gs), "jaccard": len(cs & gs) / len(u) if u else np.nan})

    between_df = pd.DataFrame(between_rows)

    # 保存每次bootstrap原始选择结果，这次以后就不用重新跑
    cell_records.to_csv(os.path.join(output_dir, "cell_specific_bootstrap_targets.csv"), index=False)
    gene_records.to_csv(os.path.join(output_dir, "gene_level_bootstrap_targets.csv"), index=False)
    run_df.to_csv(os.path.join(output_dir, "bootstrap_run_summary.csv"), index=False)

    cell_feature_freq.to_csv(os.path.join(output_dir, "cell_specific_feature_selection_frequency.csv"), index=False)
    cell_gene_freq.to_csv(os.path.join(output_dir, "cell_specific_gene_selection_frequency.csv"), index=False)
    gene_level_freq.to_csv(os.path.join(output_dir, "gene_level_gene_selection_frequency.csv"), index=False)

    cell_feature_internal.to_csv(os.path.join(output_dir, "cell_specific_feature_internal_jaccard.csv"), index=False)
    cell_gene_internal.to_csv(os.path.join(output_dir, "cell_specific_gene_internal_jaccard.csv"), index=False)
    gene_level_internal.to_csv(os.path.join(output_dir, "gene_level_gene_internal_jaccard.csv"), index=False)
    between_df.to_csv(os.path.join(output_dir, "between_strategy_gene_jaccard.csv"), index=False)

    print("\n===== Mean Jaccard =====")
    print(f"Cell-specific FEATURE internal: {cell_feature_internal['jaccard'].mean():.3f}")
    print(f"Cell-specific GENE internal:    {cell_gene_internal['jaccard'].mean():.3f}")
    print(f"Gene-level GENE internal:       {gene_level_internal['jaccard'].mean():.3f}")
    print(f"Between-strategy GENE:          {between_df['jaccard'].mean():.3f}")

    print("\n===== Cell-specific top FEATURES =====")
    print(cell_feature_freq.head(20).to_string(index=False))

    print("\n===== Gene-level top GENES =====")
    print(gene_level_freq.head(20).to_string(index=False))

    return {
        "cell_records": cell_records, "gene_records": gene_records, "run_summary": run_df,
        "cell_feature_frequency": cell_feature_freq, "cell_gene_frequency": cell_gene_freq,
        "gene_level_frequency": gene_level_freq, "cell_feature_internal_jaccard": cell_feature_internal,
        "cell_gene_internal_jaccard": cell_gene_internal, "gene_level_internal_jaccard": gene_level_internal,
        "between_gene_jaccard": between_df
    }

In [ ]:
bootstrap_result = run_bootstrap_stability(
    X=X, en_model=en_model, scaler_expr=scaler_expr, selected_expr_cols=selected_expr_cols,
    scaler_prop=scaler_prop, prop_cols=prop_cols, selected_features=selected_features,
    age_range=(50, 60), n_boot=50, reduce_years=10, seed=2026,
    output_dir=str(OUTPUT_DIR / "bootstrap_stability/50-60")
)

In [ ]:
# ============================================================
# 另外三个年龄段bootstrap stability
# ============================================================
base_out = str(OUTPUT_DIR / "bootstrap_stability")

bootstrap_results = {}

for age_range in [(40, 50), (60, 70), (70, 90)]:
    age = f"{age_range[0]}-{age_range[1]}"
    print(f"\n{'='*70}\nRunning {age} years\n{'='*70}")

    bootstrap_results[age] = run_bootstrap_stability(
        X=X, en_model=en_model, scaler_expr=scaler_expr, selected_expr_cols=selected_expr_cols,
        scaler_prop=scaler_prop, prop_cols=prop_cols, selected_features=selected_features,
        age_range=age_range, n_boot=50, reduce_years=10, seed=2026,
        output_dir=os.path.join(base_out, age)
    )

In [ ]:
# ============================================================
# 汇总四年龄段Jaccard
# ============================================================
age_order = ["40-50", "50-60", "60-70", "70-90"]
summary_rows, long_rows = [], []

for age in age_order:
    d = os.path.join(base_out, age)

    cell_feature = pd.read_csv(os.path.join(d, "cell_specific_feature_internal_jaccard.csv"))
    cell_gene = pd.read_csv(os.path.join(d, "cell_specific_gene_internal_jaccard.csv"))
    gene_internal = pd.read_csv(os.path.join(d, "gene_level_gene_internal_jaccard.csv"))
    between = pd.read_csv(os.path.join(d, "between_strategy_gene_jaccard.csv"))

    summary_rows.append({
        "age_group": age,
        "cell_specific_feature_internal": cell_feature["jaccard"].mean(),
        "cell_specific_gene_internal": cell_gene["jaccard"].mean(),
        "gene_level_gene_internal": gene_internal["jaccard"].mean(),
        "between_strategy_gene": between["jaccard"].mean()
    })

    for v in cell_feature["jaccard"].dropna(): long_rows.append({"age_group": age, "metric": "Cell-specific internal", "jaccard": v})
    for v in gene_internal["jaccard"].dropna(): long_rows.append({"age_group": age, "metric": "Gene-level internal", "jaccard": v})
    for v in between["jaccard"].dropna(): long_rows.append({"age_group": age, "metric": "Between strategies", "jaccard": v})

summary_df = pd.DataFrame(summary_rows)
jaccard_long = pd.DataFrame(long_rows)

summary_df.to_csv(os.path.join(base_out, "bootstrap_stability_summary.csv"), index=False)

print(summary_df.round(3).to_string(index=False))

In [ ]:
# ============================================================
# Bootstrap stability可视化
# ============================================================
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

font_dir = str(FONT_DIR)
for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 13, "font.weight": "bold",
    "axes.labelsize": 15, "axes.labelweight": "bold", "xtick.labelsize": 12, "ytick.labelsize": 12,
    "axes.linewidth": 1.2, "pdf.fonttype": 42, "ps.fonttype": 42
})

plot_df = summary_df.copy()
x = np.arange(len(age_order))

colors = {
    "Cell-specific internal": "#527A8C",
    "Gene-level internal": "#B66A70",
    "Between strategies": "#777777"
}

series = {
    "Cell-specific internal": "cell_specific_feature_internal",
    "Gene-level internal": "gene_level_gene_internal",
    "Between strategies": "between_strategy_gene"
}

fig, ax = plt.subplots(figsize=(6.2, 4.8))

for label, col in series.items():
    yv = plot_df.set_index("age_group").loc[age_order, col].values
    ax.plot(x, yv, marker="o", markersize=7.5, linewidth=2.2, color=colors[label], label=label)
    for xi, yi in zip(x, yv):
        ax.text(xi, yi + 0.025, f"{yi:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=colors[label])

ax.set_xticks(x)
ax.set_xticklabels([f"{a} years" for a in age_order], fontweight="bold")
ax.set_ylabel("Mean Jaccard index", fontsize=15, fontweight="bold")
ax.set_ylim(0, 1.08)
ax.tick_params(axis="both", width=1.2, length=4)
ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.2)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

leg = ax.legend(frameon=False, fontsize=11, loc="best")
for t in leg.get_texts(): t.set_fontweight("bold")

plt.tight_layout()
plt.savefig(os.path.join(base_out, "bootstrap_stability_across_age_groups.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(base_out, "bootstrap_stability_across_age_groups.png"), dpi=600, bbox_inches="tight")
plt.show()

## Functional similarity and gene-label permutations (Fig. 5g)

Source: cells 20–21 and 25–28. The original cell 20 used a global `feature_gene_map` that was not initialized in a fresh kernel. The following initialization calls the original helper from cell 5 with the already loaded selected features; no genes are invented. Original GO BP 2023 / Reactome 2022 libraries, 3–300 background-gene limits, 5,000 permutations, one-sided test and joint BH adjustment across eight comparisons are retained. Online GSEApy library access is required. Historical functional-z alternatives are excluded; the final mean-selection-frequency method is used.

In [ ]:
feature_gene_map = build_feature_gene_map(selected_features, selected_expr_cols)

In [ ]:
import os
import numpy as np
import pandas as pd
import gseapy as gp
from scipy.stats import spearmanr

base_out = str(OUTPUT_DIR / "bootstrap_stability")
func_out = os.path.join(base_out, "functional_similarity")
os.makedirs(func_out, exist_ok=True)

age_order = ["40-50", "50-60", "60-70", "70-90"]
background = sorted(pd.Series(feature_gene_map).astype(str).str.upper().unique())

def load_frequency(path):
    df = pd.read_csv(path)
    df["gene"] = df["gene"].astype(str).str.upper()
    freq = df.groupby("gene")["selection_frequency"].max()
    score = pd.Series(0.0, index=background)
    common = score.index.intersection(freq.index)
    score.loc[common] = freq.loc[common]
    return score

In [ ]:
reactome = gp.get_library(name=str(GMT_DIR / "Reactome_2022.gmt") if (GMT_DIR / "Reactome_2022.gmt").is_file() else "Reactome_2022", organism="Human")
gobp = gp.get_library(name=str(GMT_DIR / "GO_Biological_Process_2023.gmt") if (GMT_DIR / "GO_Biological_Process_2023.gmt").is_file() else "GO_Biological_Process_2023", organism="Human")

print("Reactome terms:", len(reactome))
print("GO BP terms:", len(gobp))

In [ ]:
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# ============================================================
# pathway mean selection frequency
# 不使用functional_z，避免pathway size共同缩放带来的相关性
# ============================================================
def pathway_mean_profile(gene_score, gene_sets, min_genes=3, max_genes=300):
    bg = set(gene_score.index)
    rows = []
    for pathway, genes in gene_sets.items():
        genes = list(set(map(str.upper, genes)) & bg)
        if len(genes) < min_genes or len(genes) > max_genes: continue
        rows.append({"pathway": pathway, "n_genes": len(genes), "score": gene_score.loc[genes].mean()})
    return pd.DataFrame(rows).set_index("pathway")["score"]

# ============================================================
# gene-label permutation
# ============================================================
def functional_similarity_perm(cell_score, gene_score, gene_sets, n_perm=5000, min_genes=3, max_genes=300, seed=2026):
    rng = np.random.default_rng(seed)

    p_cell = pathway_mean_profile(cell_score, gene_sets, min_genes, max_genes)
    p_gene = pathway_mean_profile(gene_score, gene_sets, min_genes, max_genes)
    common = p_cell.index.intersection(p_gene.index)

    p_cell, p_gene = p_cell.loc[common], p_gene.loc[common]
    observed_rho = spearmanr(p_cell, p_gene).statistic

    null_rho = np.empty(n_perm)
    values = gene_score.values.copy()

    for i in range(n_perm):
        shuffled = pd.Series(rng.permutation(values), index=gene_score.index)
        p_random = pathway_mean_profile(shuffled, gene_sets, min_genes, max_genes).reindex(common)
        null_rho[i] = spearmanr(p_cell, p_random).statistic

    # 单侧：真实相似性是否显著高于随机
    p_perm = (np.sum(null_rho >= observed_rho) + 1) / (n_perm + 1)

    # top20 pathway overlap
    n_top = min(20, len(common))
    top_cell = set(p_cell.nlargest(n_top).index)
    top_gene = set(p_gene.nlargest(n_top).index)
    top_shared = len(top_cell & top_gene)
    top_jaccard = top_shared / len(top_cell | top_gene)

    return observed_rho, p_perm, top_shared, top_jaccard, null_rho

In [ ]:
results = []

for age in age_order:
    cell_score = load_frequency(os.path.join(base_out, age, "cell_specific_gene_selection_frequency.csv"))
    gene_score = load_frequency(os.path.join(base_out, age, "gene_level_gene_selection_frequency.csv"))

    for library, gene_sets in [("Reactome", reactome), ("GO_BP", gobp)]:
        print(f"Running {age} | {library}")

        rho, p_perm, top_shared, top_jaccard, null = functional_similarity_perm(
            cell_score, gene_score, gene_sets,
            n_perm=5000, min_genes=3, max_genes=300, seed=2026
        )

        results.append({
            "age_group": age, "library": library,
            "spearman_rho": rho, "permutation_p": p_perm,
            "top20_shared": top_shared, "top20_jaccard": top_jaccard,
            "null_rho_mean": np.mean(null), "null_rho_sd": np.std(null)
        })

similarity_perm_df = pd.DataFrame(results)

# 8个检验一起BH校正
similarity_perm_df["permutation_FDR"] = multipletests(
    similarity_perm_df["permutation_p"], method="fdr_bh"
)[1]

similarity_perm_df.to_csv(
    os.path.join(func_out, "functional_similarity_permutation_final.csv"),
    index=False
)

print("\n")
print(similarity_perm_df.round(3).to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

base_out = str(OUTPUT_DIR / "bootstrap_stability")
func_file = os.path.join(base_out, "functional_similarity", "functional_similarity_permutation_final.csv")
out_pdf = os.path.join(base_out, "perturbation_strategy_comparison.pdf")
out_png = os.path.join(base_out, "perturbation_strategy_comparison.png")

age_order = ["40-50", "50-60", "60-70", "70-90"]

# ============================================================
# 1. Arial
# ============================================================
font_dir = str(FONT_DIR)
for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 11.5, "font.weight": "bold",
    "axes.labelsize": 13.5, "axes.labelweight": "bold", "axes.linewidth": 1.2,
    "xtick.labelsize": 10.5, "ytick.labelsize": 10.5, "legend.fontsize": 9.5,
    "pdf.fonttype": 42, "ps.fonttype": 42
})

# ============================================================
# 2. Panel A：读取Jaccard
# ============================================================
jaccard_rows = []

for age in age_order:
    d = os.path.join(base_out, age)

    cf = pd.read_csv(os.path.join(d, "cell_specific_feature_internal_jaccard.csv"))["jaccard"].dropna()
    gi = pd.read_csv(os.path.join(d, "gene_level_gene_internal_jaccard.csv"))["jaccard"].dropna()
    bt = pd.read_csv(os.path.join(d, "between_strategy_gene_jaccard.csv"))["jaccard"].dropna()

    jaccard_rows += [
        {"age": age, "metric": "Cell-type-specific internal", "mean": cf.mean()},
        {"age": age, "metric": "Gene-level internal", "mean": gi.mean()},
        {"age": age, "metric": "Between strategies", "mean": bt.mean()}
    ]

jaccard_df = pd.DataFrame(jaccard_rows)

# ============================================================
# 3. Panel B：读取functional similarity
# ============================================================
func_df = pd.read_csv(func_file)
func_df["age_group"] = pd.Categorical(func_df["age_group"], categories=age_order, ordered=True)
func_df = func_df.sort_values(["age_group", "library"])

# ============================================================
# 4. 配色
# ============================================================
colors_a = {
    "Cell-type-specific internal": "#527A8C",
    "Gene-level internal": "#B66A70",
    "Between strategies": "#777777"
}

colors_b = {
    "GO_BP": "#6F7F56",
    "Reactome": "#8B7B96"
}

# ============================================================
# 5. 作图
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.65), gridspec_kw={"wspace": 0.34})
ax1, ax2 = axes
x = np.arange(len(age_order))

# ---------------- Panel A ----------------
offsets_a = [-0.17, 0, 0.17]

for offset, metric in zip(offsets_a, colors_a):
    tmp = jaccard_df[jaccard_df["metric"] == metric].set_index("age").loc[age_order]
    y = tmp["mean"].values
    xp = x + offset

    ax1.scatter(xp, y, s=52, color=colors_a[metric], edgecolor="white", linewidth=0.8, zorder=3, label=metric)

    for xi, yi in zip(xp, y):
        ax1.text(xi, yi + 0.035, f"{yi:.2f}", ha="center", va="bottom",
                 fontsize=8.8, fontweight="bold", color=colors_a[metric])

ax1.set_xticks(x)
ax1.set_xticklabels(age_order, fontweight="bold")
ax1.set_xlabel("Predicted age group", fontweight="bold")
ax1.set_ylabel("Mean Jaccard index", fontweight="bold")
ax1.set_ylim(0, 1.08)
ax1.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.18)
ax1.tick_params(width=1.1, length=3.5)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

leg1 = ax1.legend(frameon=False, loc="lower left", bbox_to_anchor=(0.0, 0.02),
                  ncol=1, handletextpad=0.4, borderaxespad=0)
for t in leg1.get_texts(): t.set_fontweight("bold")

ax1.text(-0.16, 1.04, "A", transform=ax1.transAxes, fontsize=15, fontweight="bold")

# ---------------- Panel B ----------------
offsets_b = {"GO_BP": -0.08, "Reactome": 0.08}
labels_b = {"GO_BP": "GO biological process", "Reactome": "Reactome"}

for library in ["GO_BP", "Reactome"]:
    tmp = func_df[func_df["library"] == library].set_index("age_group").loc[age_order]
    y = tmp["spearman_rho"].values
    xp = x + offsets_b[library]

    # 实心 = permutation FDR < 0.05；空心 = nonsignificant
    sig = tmp["permutation_FDR"].values < 0.05

    ax2.scatter(xp[sig], y[sig], s=58, color=colors_b[library], edgecolor="white",
                linewidth=0.8, zorder=3, label=labels_b[library])

    ax2.scatter(xp[~sig], y[~sig], s=58, facecolor="white", edgecolor=colors_b[library],
                linewidth=1.6, zorder=3)

    for xi, yi, fdr in zip(xp, y, tmp["permutation_FDR"].values):
        star = "*" if fdr < 0.05 else ""
        ax2.text(xi, yi + 0.035, f"{yi:.2f}{star}", ha="center", va="bottom",
                 fontsize=8.8, fontweight="bold", color=colors_b[library])

ax2.axhline(0, color="#999999", linewidth=0.8, linestyle="--", zorder=1)
ax2.set_xticks(x)
ax2.set_xticklabels(age_order, fontweight="bold")
ax2.set_xlabel("Predicted age group", fontweight="bold")
ax2.set_ylabel("Functional similarity (Spearman ρ)", fontweight="bold")
ax2.set_ylim(-0.08, 0.78)
ax2.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.18)
ax2.tick_params(width=1.1, length=3.5)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

handles, labels = ax2.get_legend_handles_labels()
leg2 = ax2.legend(handles[:2], labels[:2], frameon=False, loc="upper left")
for t in leg2.get_texts(): t.set_fontweight("bold")

ax2.text(0.98, 0.03, "* Permutation FDR < 0.05", transform=ax2.transAxes,
         ha="right", va="bottom", fontsize=8.5, fontweight="bold", color="#555555")

ax2.text(-0.16, 1.04, "B", transform=ax2.transAxes, fontsize=15, fontweight="bold")

plt.tight_layout()
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(out_png, dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

base_out = str(OUTPUT_DIR / "bootstrap_stability")
func_file = os.path.join(base_out, "functional_similarity", "functional_similarity_permutation_final.csv")
out_pdf = os.path.join(base_out, "GOBP_functional_similarity.pdf")
out_png = os.path.join(base_out, "GOBP_functional_similarity.png")
age_order = ["40-50", "50-60", "60-70", "70-90"]

font_dir = str(FONT_DIR)
for f in os.listdir(font_dir):
    if f.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, f))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 12, "font.weight": "bold",
    "axes.labelsize": 15, "axes.labelweight": "bold", "axes.linewidth": 1.2,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
    "pdf.fonttype": 42, "ps.fonttype": 42
})

df = pd.read_csv(func_file)
df = df[df["library"] == "GO_BP"].copy()
df["age_group"] = pd.Categorical(df["age_group"], categories=age_order, ordered=True)
df = df.sort_values("age_group")

x = np.arange(len(age_order))
y = df.set_index("age_group").loc[age_order, "spearman_rho"].values
fdr = df.set_index("age_group").loc[age_order, "permutation_FDR"].values
sig = fdr < 0.05

fig, ax = plt.subplots(figsize=(4.1, 3.8))
color = "#6F7F56"

ax.scatter(x[sig], y[sig], s=82, color=color, edgecolor="white", linewidth=0.9, zorder=3)
ax.scatter(x[~sig], y[~sig], s=82, facecolor="white", edgecolor=color, linewidth=1.8, zorder=3)

for xi, yi, fi in zip(x, y, fdr):
    lab = f"{yi:.2f}" + ("*" if fi < 0.05 else "")
    ax.text(xi, yi + 0.035, lab, ha="center", va="bottom", fontsize=11, fontweight="bold", color=color)

ax.axhline(0, color="#9A9A9A", linewidth=0.8, linestyle="--", zorder=1)
ax.set_xticks(x)
ax.set_xticklabels([f"{a}y" for a in age_order], fontweight="bold")
ax.set_xlabel("Predicted age group", fontweight="bold")
ax.set_ylabel("Functional similarity\n(Spearman ρ)", fontweight="bold")
ax.set_ylim(-0.05, 0.78)
ax.set_xlim(-0.38, len(age_order) - 0.62)
ax.tick_params(axis="both", width=1.2, length=4)
ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(0.98, 0.04, "* Permutation FDR < 0.05", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=10, fontweight="bold", color="#555555")

plt.tight_layout(pad=0.5)
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(out_png, dpi=600, bbox_inches="tight")
plt.show()